#  Loan Approval Classification – Regresión Logística

**Objetivo:** Predecir si un préstamo será aprobado o no, usando variables financieras del solicitante.  
**Dataset:** Loan Approval Dataset (Kaggle)  
**Modelo:** Regresión Logística  

---
**Flujo del proyecto:**
1. Carga y exploración de datos (EDA)
2. Limpieza y preprocesamiento
3. Codificación de variables categóricas
4. Entrenamiento del modelo
5. Evaluación de métricas
6. Análisis de importancia de variables
7. Predicción sobre nuevos solicitantes

## 1. Importar librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report,
    roc_curve, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

# Estilo de gráficas
sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.figsize'] = (10, 5)

print('Librerías cargadas correctamente')

## 2. Carga de datos

In [ ]:
# Cargar el dataset
df = pd.read_csv('../data/raw/loan_approval_dataset.csv')

print(f'Shape del dataset: {df.shape}')
print(f'Columnas: {list(df.columns)}')
df.head()

## 3. Exploración de Datos (EDA)

In [ ]:
# Información general del dataset
print('=== TIPOS DE VARIABLES ===')
print(df.dtypes)
print()
print('=== VALORES NULOS ===')
print(df.isnull().sum())

In [ ]:
# Estadísticas descriptivas
df.describe()

In [ ]:
# Distribución de la variable objetivo
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Conteo
df['loan_approved'].value_counts().plot(kind='bar', ax=axes[0], color=['#d9534f', '#5bc0de'])
axes[0].set_title('Distribución de loan_approved')
axes[0].set_xlabel('Aprobado (1) / Rechazado (0)')
axes[0].set_ylabel('Cantidad')
axes[0].tick_params(rotation=0)

# Porcentaje
df['loan_approved'].value_counts(normalize=True).plot(kind='pie', ax=axes[1],
    labels=['Rechazado', 'Aprobado'], autopct='%1.1f%%', colors=['#d9534f', '#5bc0de'])
axes[1].set_title('Proporción de aprobaciones')

plt.tight_layout()
plt.show()

print(df['loan_approved'].value_counts())

In [ ]:
# Distribución de variables numéricas
num_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c != 'loan_approved']

df[num_cols].hist(bins=30, figsize=(14, 8), color='steelblue', edgecolor='white')
plt.suptitle('Distribución de Variables Numéricas', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlación con la variable objetivo
corr = df[num_cols + ['loan_approved']].corr()[['loan_approved']].sort_values('loan_approved', ascending=False)

plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.3f', cmap='coolwarm', center=0,
            linewidths=0.5, cbar_kws={'label': 'Correlación'})
plt.title('Correlación de variables con loan_approved')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplots: variables clave por resultado
key_vars = ['income', 'credit_score', 'loan_amount', 'years_employed']
key_vars = [v for v in key_vars if v in df.columns]

fig, axes = plt.subplots(1, len(key_vars), figsize=(14, 5))
for i, col in enumerate(key_vars):
    sns.boxplot(data=df, x='loan_approved', y=col, ax=axes[i],
                palette={0: '#d9534f', 1: '#5bc0de'})
    axes[i].set_title(f'{col} por resultado')
    axes[i].set_xlabel('Aprobado')

plt.suptitle('Variables financieras según aprobación del préstamo', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

## 4. Preprocesamiento

In [ ]:
# Copiar dataframe de trabajo
df_model = df.copy()

# Convertir target a binario si es necesario
if df_model['loan_approved'].dtype == object:
    df_model['loan_approved'] = df_model['loan_approved'].map({'Y': 1, 'N': 0, 'Yes': 1, 'No': 0})

# Identificar columnas categóricas (excluyendo target)
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print(f'Columnas categóricas: {cat_cols}')

# One-hot encoding
if cat_cols:
    df_model = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)
    print(f'Shape después de encoding: {df_model.shape}')

# Eliminar columnas con demasiada cardinalidad (>500 columnas generadas por una sola variable)
print(f'Total de columnas para el modelo: {df_model.shape[1]}')

In [ ]:
# Separar features y target
X = df_model.drop('loan_approved', axis=1)
y = df_model['loan_approved']

print(f'X shape: {X.shape}')
print(f'y distribución:\n{y.value_counts()}')

In [ ]:
# División train/test (80/20 estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} registros')
print(f'Test:  {X_test.shape[0]} registros')

In [ ]:
# Escalado de variables numéricas
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(' Escalado aplicado (StandardScaler)')

## 5. Entrenamiento del Modelo

In [ ]:
# Modelo de Regresión Logística
model = LogisticRegression(
    max_iter=1000,
    solver='lbfgs',
    C=1.0,          # Regularización inversa
    random_state=42
)

model.fit(X_train_scaled, y_train)
print('Modelo entrenado correctamente')

## 6. Evaluación del Modelo

In [ ]:
# Predicciones
y_pred  = model.predict(X_test_scaled)
y_proba = model.predict_proba(X_test_scaled)[:, 1]

# Métricas
print('='*45)
print('        MÉTRICAS DEL MODELO (Test Set)')
print('='*45)
print(f'  Accuracy :  {accuracy_score(y_test, y_pred):.4f}')
print(f'  Precision:  {precision_score(y_test, y_pred):.4f}')
print(f'  Recall   :  {recall_score(y_test, y_pred):.4f}')
print(f'  F1-Score :  {f1_score(y_test, y_pred):.4f}')
print(f'  ROC-AUC  :  {roc_auc_score(y_test, y_proba):.4f}')
print('='*45)
print()
print(classification_report(y_test, y_pred, target_names=['Rechazado', 'Aprobado']))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Rechazado', 'Aprobado'],
            yticklabels=['Rechazado', 'Aprobado'])
plt.title('Matriz de Confusión')
plt.ylabel('Real')
plt.xlabel('Predicción')
plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
auc = roc_auc_score(y_test, y_proba)

plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC AUC = {auc:.4f}')
plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random classifier')
plt.xlabel('Tasa de Falsos Positivos')
plt.ylabel('Tasa de Verdaderos Positivos')
plt.title('Curva ROC – Regresión Logística')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 7. Importancia de Variables

In [ ]:
# Coeficientes del modelo
coef_df = pd.DataFrame({
    'variable': X.columns,
    'coeficiente': model.coef_[0]
}).sort_values('coeficiente', key=abs, ascending=False)

# Top 15 variables más importantes
top15 = coef_df.head(15)

plt.figure(figsize=(10, 6))
colors = ['#d9534f' if c < 0 else '#5bc0de' for c in top15['coeficiente']]
sns.barplot(data=top15, x='coeficiente', y='variable', palette=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 Variables más importantes (Coeficientes)')
plt.xlabel('Coeficiente (magnitud = importancia)')
plt.tight_layout()
plt.show()

print('\nTop 10 variables:')
print(coef_df.head(10).to_string(index=False))

## 8. Análisis sin la variable `points`

Como se documentó, `points` tiene un poder predictivo casi perfecto y podría ser un proxy del target.  
Aquí evaluamos el modelo **sin** esa variable para ver cómo se comportan las variables financieras reales.

In [ ]:
if 'points' in X.columns:
    X_no_points = X.drop(columns=['points'])

    X_train2, X_test2, y_train2, y_test2 = train_test_split(
        X_no_points, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler2 = StandardScaler()
    X_train2_s = scaler2.fit_transform(X_train2)
    X_test2_s  = scaler2.transform(X_test2)

    model2 = LogisticRegression(max_iter=1000, random_state=42)
    model2.fit(X_train2_s, y_train2)
    y_pred2 = model2.predict(X_test2_s)

    print('=== MODELO SIN VARIABLE `points` ===')
    print(f'Accuracy : {accuracy_score(y_test2, y_pred2):.4f}')
    print(f'F1-Score : {f1_score(y_test2, y_pred2):.4f}')
    print(f'ROC-AUC  : {roc_auc_score(y_test2, model2.predict_proba(X_test2_s)[:,1]):.4f}')
else:
    print('La variable `points` no está en el dataset.')

## 9. Predicción sobre un nuevo solicitante

In [ ]:
# Ejemplo de predicción para un nuevo solicitante
# (Ajustar los valores según las columnas reales del dataset)

nuevo_solicitante = pd.DataFrame([{
    col: 0 for col in X.columns  # base en ceros
}])

# Llenar con valores reales de ejemplo
if 'income' in nuevo_solicitante.columns:         nuevo_solicitante['income'] = 55000
if 'credit_score' in nuevo_solicitante.columns:   nuevo_solicitante['credit_score'] = 720
if 'loan_amount' in nuevo_solicitante.columns:    nuevo_solicitante['loan_amount'] = 15000
if 'years_employed' in nuevo_solicitante.columns: nuevo_solicitante['years_employed'] = 5
if 'points' in nuevo_solicitante.columns:         nuevo_solicitante['points'] = 80

# Escalar y predecir
nuevo_scaled = scaler.transform(nuevo_solicitante)
prob = model.predict_proba(nuevo_scaled)[0, 1]
resultado = 'APROBADO ' if prob >= 0.5 else 'RECHAZADO '

print(f'Probabilidad de aprobación: {prob:.2%}')
print(f'Resultado: {resultado}')

## 10. Conclusiones

- El modelo de **Regresión Logística** logró alta precisión en la clasificación de solicitudes de préstamo.
- La variable **`points`** domina el poder predictivo del modelo, lo cual sugiere que es un score crediticio derivado del target.
- Al remover `points`, las variables financieras como **income**, **credit_score** y **loan_amount** se vuelven las principales predictoras — más representativo de un escenario real.
- El modelo es interpretable mediante coeficientes: un coeficiente positivo indica que la variable **aumenta** la probabilidad de aprobación.

### Posibles mejoras futuras
- Probar modelos como **Random Forest** o **XGBoost** para capturar relaciones no lineales.
- Aplicar **regularización L1 (Lasso)** para selección automática de variables.
- Reducir dimensionalidad agrupando ciudades con baja frecuencia.
- Evaluar con **validación cruzada** (k-fold) para mayor robustez.